# TP 4 - Premiers traitements PySpark sur CSV et Parquet

**Module :** Big Data (Hadoop & Spark)
**Séance :** Prise en main d'Apache Spark
**Durée :** 50 minutes
**Enseignant :** Mouaad AGOURRAM, Institut GoldenCollar

## Contexte

Vous travaillez sur les trajets des taxis jaunes de New York de février 2024, publiés par la Taxi and
Limousine Commission (TLC) au format Parquet, et sur la table des zones de taxi de la TLC, au format CSV.

Source : <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>
Dictionnaire des données :
<https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf>

Le cluster du module tourne dans Docker sur votre machine. Ce notebook envoie ses DataFrames au serveur
Spark Connect du cluster (port 15002), où tourne le driver ; les executors tournent sur les trois workers.

## Objectifs

À la fin du TP, vous saurez ouvrir une session Spark sur le cluster, lire un fichier CSV et un fichier
Parquet, reconnaître dans la Spark UI une transformation et une action, et écrire un résultat en Parquet et
en CSV.

## Consignes

- Le cluster tourne (`docker compose up -d` à la racine du module). Depuis la racine du module,
  `uv sync --group tp`, puis ouvrez ce notebook depuis le dossier `seance4/tp` avec le noyau `.venv`.
- Gardez ouverte la Spark UI du cluster : <http://localhost:4040>.
- Exécutez les questions dans l'ordre et conservez les sorties demandées dans le notebook.
- Tous les calculs se font avec Spark : n'utilisez pas `pandas`.
- La préparation définit `cluster(commande)`, qui exécute une commande shell dans le conteneur client du
  cluster, où est installé `hdfs`, et renvoie la liste des lignes de sa sortie.
- Le bonus est facultatif.

## Préparation

Exécutez cette cellule sans la modifier. Elle télécharge dans `data/` les trajets de février 2024 (50 Mo) et
la table des zones, s'ils sont absents, puis les dépose dans un dossier `/tp4` vide de HDFS.

In [ ]:
import os
import subprocess
import urllib.request
from pathlib import Path

os.environ["GRPC_ENABLE_FORK_SUPPORT"] = "false"   # pas de message de gRPC quand cluster() lance docker

COMPOSE = Path.cwd().parents[1] / "docker-compose.yml"
assert COMPOSE.exists(), f"{COMPOSE} introuvable : ouvrez ce notebook depuis le dossier seance4/tp"


def cluster(commande):
    # Exécute une commande shell dans le conteneur client du cluster ; renvoie ses lignes de sortie
    resultat = subprocess.run(
        ["docker", "compose", "-f", str(COMPOSE), "exec", "-T", "client", "bash", "-c", commande],
        capture_output=True, text=True)
    if resultat.returncode != 0:
        raise RuntimeError(f"Échec de : {commande}\n{resultat.stderr}")
    return resultat.stdout.splitlines()


TLC = "https://d37ci6vzurychx.cloudfront.net"
FICHIERS = {"fevrier.parquet": f"{TLC}/trip-data/yellow_tripdata_2024-02.parquet",
            "taxi_zone_lookup.csv": f"{TLC}/misc/taxi_zone_lookup.csv"}
Path("data").mkdir(exist_ok=True)
for nom, url in FICHIERS.items():
    if not Path("data", nom).exists():
        urllib.request.urlretrieve(url, Path("data", nom))
    print(nom, Path("data", nom).stat().st_size, "octets")

cluster("rm -rf /tmp/tp4 && mkdir -p /tmp/tp4")
for nom in FICHIERS:
    subprocess.run(["docker", "compose", "-f", str(COMPOSE), "cp", f"data/{nom}", f"client:/tmp/tp4/{nom}"],
                   check=True, capture_output=True)
cluster("hdfs dfs -rm -r -f -skipTrash /tp4 && hdfs dfs -mkdir -p /tp4 "
        "&& hdfs dfs -put /tmp/tp4/* /tp4/ && rm -rf /tmp/tp4")
print("\n".join(cluster("hdfs dfs -ls /tp4")))

## Question 1 - Ouvrir la session Spark (3 min)

Créez une `SparkSession` nommée `spark`, connectée au serveur Spark Connect du cluster :
`sc://localhost:15002`.

**Livrable vérifiable :** affichez la version de Spark.

In [ ]:
# Votre code

## Question 2 - Lire le CSV des zones de trois façons (8 min)

Le fichier `/tp4/taxi_zone_lookup.csv` de HDFS décrit les zones de taxi. Lisez-le trois fois :

1. sans aucune option ;
2. avec l'option `header` ;
3. avec les options `header` et `inferSchema`.

Pour chaque lecture, affichez le schéma et comptez les lignes. Gardez la troisième lecture dans une variable
`zones`.

**Livrable vérifiable :** complétez le tableau ci-dessous.

| Lecture | Nom de la première colonne | Type de la première colonne | Nombre de lignes |
|---|---|---|---|
| sans option | | | |
| `header` | | | |
| `header` et `inferSchema` | | | |

In [ ]:
# Votre code

## Question 3 - Lire les trajets de février en Parquet (7 min)

Lisez le fichier `/tp4/fevrier.parquet` de HDFS dans une variable `trajets`. Affichez son schéma, puis
comptez ses colonnes et ses trajets.

**Livrable vérifiable :** le nombre de colonnes et le nombre de trajets.

In [ ]:
# Votre code

## Question 4 - Filtrer, sélectionner, puis compter (10 min)

Ouvrez la Spark UI (<http://localhost:4040>), onglet **Jobs**, et notez le nombre de jobs terminés.

1. Dans une cellule, gardez les trajets dont `trip_distance` et `total_amount` sont strictement positifs,
   puis les colonnes `PULocationID`, `payment_type`, `trip_distance` et `total_amount`. Appelez le résultat
   `valides`. Exécutez la cellule, puis relevez le nombre de jobs terminés.
2. Dans une autre cellule, comptez les lignes de `valides`. Relevez de nouveau le nombre de jobs terminés.

**Livrable vérifiable :** le nombre de trajets de `valides`, et le tableau ci-dessous complété.

| Cellule exécutée | Nouveaux jobs dans l'onglet Jobs |
|---|---|
| Filtre et sélection | |
| `count()` | |

In [ ]:
# Votre code

In [ ]:
# Votre code

## Question 5 - Compter les trajets par type de paiement (8 min)

À partir de `valides`, comptez les trajets pour chaque valeur de `payment_type`, et triez le résultat par
nombre de trajets décroissant. Appelez le résultat `par_paiement` et affichez-le.

À l'aide du dictionnaire des données, donnez le sens du type de paiement le plus fréquent.

**Livrable vérifiable :** le tableau affiché, et le sens du type le plus fréquent.

In [ ]:
# Votre code

## Question 6 - Écrire en Parquet et en CSV (10 min)

Écrivez `valides` dans HDFS, en mode `overwrite`, deux fois :

- en Parquet, dans `/tp4/sorties/parquet` ;
- en CSV avec une ligne d'en-tête, dans `/tp4/sorties/csv`.

Avec `cluster()` et la commande `hdfs dfs -du -s`, mesurez la taille de chacun des deux dossiers.

**Livrable vérifiable :** les deux tailles en octets, et le rapport entre la taille du CSV et celle du Parquet.

In [ ]:
# Votre code

## Bonus - La zone de prise en charge la plus fréquente (4 min)

Trouvez, dans `valides`, la valeur de `PULocationID` la plus fréquente. Retrouvez ensuite, avec un filtre sur
`zones`, le nom de cette zone et son arrondissement (`Borough`).

**Livrable vérifiable :** l'identifiant de la zone, son nombre de trajets, son nom et son arrondissement.

In [ ]:
# Votre code

## Fin du TP

Arrêtez la session Spark. Vérifiez ensuite que le notebook contient toutes les sorties demandées.

In [ ]:
# Arrêtez la session Spark.